# 04. Análisis de errores del módulo clásico (T44)

Qué imágenes falla el clásico, agrupadas por tipo de error (real → predicha), y **qué regla del árbol** las decidió.

Los datos se generan con `python -m src.error_analysis` a partir de `results/clasico/descriptores_*.csv` y `results/clasico/umbrales.json` (T43). Cada error se explica con la misma rama de `classic.predict`, con el descriptor que compara, su valor, el umbral y el margen (valor − umbral).

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLASSIC = ROOT / 'results' / 'clasico'
FIGURES = ROOT / 'results' / 'figuras'

## 1. Errores por par real → predicho

In [ ]:
pd.read_csv(CLASSIC / 'errores_resumen_test.csv')

In [ ]:
pd.read_csv(CLASSIC / 'errores_resumen_camara_no_vista.csv')

## 2. Ejemplos de cada tipo de error (cámara no vista)

Cada fila de la figura es un tipo de error (real → predicha) con hasta tres imágenes.

In [ ]:
display(Image(filename=str(FIGURES / 'errores_ejemplos.png')))

## 3. Causa de cada tipo de error (cámara no vista)

Cada error se explica con la rama del árbol que lo decidió. La tabla sale directo de `errores_camara_no_vista.csv`, así que los números no se escriben a mano: `margen` = valor − umbral del descriptor decisivo. En `girada → normal` y `desenfocada → normal` el descriptor decisivo es `angle` y el margen es negativo: es lo que le faltó al ángulo para superar `angle_max` y marcar la imagen como girada.

In [ ]:
errors = pd.read_csv(CLASSIC / 'errores_camara_no_vista.csv')
causes = (
    errors.assign(par=errors['label'] + ' → ' + errors['pred'])
    .groupby(['par', 'regla', 'descriptor_decisivo'])
    .agg(n=('path', 'size'), valor_mediano=('valor', 'median'), valor_max=('valor', 'max'),
         umbral=('umbral', 'first'), margen_mediano=('margen', 'median'))
    .round(3)
    .reset_index()
    .sort_values(['par', 'n'], ascending=[True, False])
)
causes

Lectura: el árbol compara primero las coincidencias ORB con la referencia y después el ángulo. Los paneos pequeños (`girada → normal`) conservan coincidencias normales y su ángulo queda por debajo de `angle_max`, por eso el árbol los acepta como la misma vista. Estas causas salen de las ramas del árbol, no de comparar contra la mediana; antes de presentarlas en la defensa, mira las imágenes de la figura para confirmarlas.


In [ ]:
pd.read_csv(CLASSIC / 'errores_camara_no_vista.csv').head(10)